# 01 — Exploración, ETL y entrenamiento

Notebook para Databricks. Requiere el CSV original en el Volume `workspace.default.telco_churn`. Las escrituras en modo overwrite reemplazan tablas existentes.

## Ingesta y exploración

In [ ]:

# PASO 2: LEER DATOS CON PYSPARK

# Ruta del archivo CSV
ruta = "/Volumes/workspace/default/telco_churn/WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Leer el archivo
df = spark.read.csv(
    ruta,
    header=True,
    inferSchema=True
)

# Mostrar los primeros registros
display(df.limit(10))


In [ ]:

# Contar clientes
print("Total de registros:", df.count())

# Contar columnas
print("Total de columnas:", len(df.columns))

# Mostrar estructura
df.printSchema()


In [ ]:

print("Total de registros:", df.count())
print("Total de columnas:", len(df.columns))

df.printSchema()


## Bronze — almacenamiento Delta

In [ ]:

# Guardar los datos originales en formato Delta

df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.bronze_telco")

print("Tabla Bronze creada correctamente")


CREACION DE TABLA DELTA

In [ ]:

# Consultar nuestra primera tabla Delta

df_bronze = spark.table(
    "workspace.default.bronze_telco"
)

print("Registros en Bronze:", df_bronze.count())

display(df_bronze.limit(10))


## Silver — limpieza y calidad

In [ ]:

# PASO 4 - CAPA SILVER

from pyspark.sql import functions as F

# Leer datos desde Bronze
df_bronze = spark.table(
    "workspace.default.bronze_telco"
)

print("Registros originales:", df_bronze.count())

display(df_bronze.limit(5))


In [ ]:

# Revisar valores nulos por columna

df_nulos = df_bronze.select([
    F.sum(
        F.when(
            F.col(c).isNull() |
            (F.trim(F.col(c).cast("string")) == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df_bronze.columns
])

display(df_nulos)


In [ ]:

# LIMPIEZA DE DATOS CON PYSPARK

df_silver = (
    df_bronze

    # Convertir TotalCharges a número decimal
    .withColumn(
        "TotalCharges",
        F.when(
            F.trim(F.col("TotalCharges").cast("string")) == "",
            None
        ).otherwise(
            F.col("TotalCharges").cast("double")
        )
    )

    # Convertir cargos mensuales a decimal
    .withColumn(
        "MonthlyCharges",
        F.col("MonthlyCharges").cast("double")
    )

    # Convertir antigüedad a entero
    .withColumn(
        "tenure",
        F.col("tenure").cast("int")
    )

    # Crear variable objetivo numérica
    .withColumn(
        "ChurnLabel",
        F.when(F.col("Churn") == "Yes", 1).otherwise(0)
    )

    # Eliminar duplicados por identificador
    .dropDuplicates(["customerID"])

    # Eliminar filas sin identificador
    .filter(
        F.col("customerID").isNotNull() &
        (F.trim(F.col("customerID")) != "")
    )
)

print("Registros después de limpieza:", df_silver.count())

display(df_silver.limit(10))


In [ ]:

# Imputar TotalCharges solo para clientes nuevos

df_silver = df_silver.withColumn(
    "TotalCharges",
    F.when(
        F.col("TotalCharges").isNull() &
        (F.col("tenure") == 0),
        F.lit(0.0)
    ).otherwise(F.col("TotalCharges"))
)

# Comprobar valores nulos restantes
df_silver.select(
    F.sum(
        F.col("TotalCharges").isNull().cast("int")
    ).alias("Nulos_TotalCharges")
).show()


In [ ]:

# GUARDAR CAPA SILVER EN DELTA LAKE

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_telco")

print("Tabla Silver creada correctamente")

# Verificar registros guardados
df_verificacion = spark.table(
    "workspace.default.silver_telco"
)

print("Total Silver:", df_verificacion.count())

display(df_verificacion.limit(10))


In [ ]:

# Revisar valores vacios en TotalCharges

df_bronze.select(
    F.count(
        F.when(
            F.col("TotalCharges").isNull() |
            (F.trim(F.col("TotalCharges").cast("string")) == ""),
            1
        )
    ).alias("Valores_Vacios_TotalCharges")
).show()


In [ ]:

from pyspark.sql import functions as F

# Leer los datos originales
df_bronze = spark.table(
    "workspace.default.bronze_telco"
)

# Crear la capa Silver
df_silver = (
    df_bronze

    # Convertir TotalCharges a decimal de forma segura
    .withColumn(
        "TotalCharges",
        F.expr("try_cast(TotalCharges AS DOUBLE)")
    )

    # Reemplazar nulos por cero si tenure es 0
    .withColumn(
        "TotalCharges",
        F.when(
            F.col("TotalCharges").isNull() &
            (F.col("tenure") == 0),
            F.lit(0.0)
        ).otherwise(F.col("TotalCharges"))
    )

    # Crear variable objetivo para Machine Learning
    .withColumn(
        "ChurnLabel",
        F.when(F.col("Churn") == "Yes", 1)
         .otherwise(0)
    )

    # Eliminar clientes duplicados
    .dropDuplicates(["customerID"])
)

# Mostrar resultados
print("Registros Silver:", df_silver.count())

display(df_silver.limit(10))


In [ ]:

# Verificar calidad de los datos Silver

print("Total de clientes:", df_silver.count())

# Revisar valores nulos en TotalCharges
df_silver.select(
    F.sum(
        F.col("TotalCharges").isNull().cast("int")
    ).alias("Nulos_TotalCharges")
).show()

# Revisar clientes que abandonaron
display(
    df_silver.groupBy("Churn", "ChurnLabel")
    .count()
    .orderBy("ChurnLabel")
)


In [ ]:

# Guardar los datos limpios en Delta Lake

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_telco")

print("CAPA SILVER CREADA CORRECTAMENTE")


In [ ]:

# Leer la tabla Silver desde Unity Catalog

df_verificacion = spark.table(
    "workspace.default.silver_telco"
)

print("Registros guardados:", df_verificacion.count())

display(df_verificacion.limit(5))


## Gold — preparación y análisis

In [ ]:

# PASO 5 - CREAR CAPA GOLD

from pyspark.sql import functions as F

# Leer datos limpios desde Silver
df_silver = spark.table(
    "workspace.default.silver_telco"
)

# Crear tabla Gold con información analítica
df_gold = df_silver.select(
    "customerID",
    "gender",
    "SeniorCitizen",
    "tenure",
    "Contract",
    "InternetService",
    "PaymentMethod",
    "MonthlyCharges",
    "TotalCharges",
    "ChurnLabel"
)

# Guardar tabla Delta Gold
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_telco")

print("CAPA GOLD CREADA CORRECTAMENTE")
print("Total de clientes:", df_gold.count())


In [ ]:

# Calcular indicadores generales

df_indicadores = spark.sql("""
SELECT
    COUNT(*) AS total_clientes,
    SUM(ChurnLabel) AS clientes_abandonaron,
    COUNT(*) - SUM(ChurnLabel) AS clientes_permanecen,
    ROUND(AVG(ChurnLabel) * 100, 2) AS porcentaje_abandono
FROM workspace.default.gold_telco
""")

display(df_indicadores)


In [ ]:

# Analizar abandono por contrato

df_contratos = spark.sql("""
SELECT
    Contract AS tipo_contrato,
    COUNT(*) AS total_clientes,
    SUM(ChurnLabel) AS abandonaron,
    ROUND(AVG(ChurnLabel) * 100, 2) AS porcentaje_abandono
FROM workspace.default.gold_telco
GROUP BY Contract
ORDER BY porcentaje_abandono DESC
""")

display(df_contratos)


In [ ]:

# Analizar abandono por antigüedad

df_antiguedad = spark.sql("""
SELECT
    CASE
        WHEN tenure <= 12 THEN '0-12 meses'
        WHEN tenure <= 24 THEN '13-24 meses'
        WHEN tenure <= 48 THEN '25-48 meses'
        ELSE '49+ meses'
    END AS grupo_antiguedad,
    COUNT(*) AS total_clientes,
    ROUND(AVG(ChurnLabel) * 100, 2) AS porcentaje_abandono
FROM workspace.default.gold_telco
GROUP BY 1
ORDER BY
    CASE grupo_antiguedad
        WHEN '0-12 meses' THEN 1
        WHEN '13-24 meses' THEN 2
        WHEN '25-48 meses' THEN 3
        ELSE 4
    END
""")

display(df_antiguedad)


In [ ]:

# Guardar tablas analíticas Gold

df_indicadores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_indicadores")

df_contratos.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_contratos")

df_antiguedad.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_antiguedad")

print("TABLAS ANALITICAS GOLD GUARDADAS")


REGRESIÓN LOGÍSTICA

## Entrenamiento y evaluación

In [ ]:

# PASO 6 - MACHINE LEARNING CON SPARK

from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    VectorAssembler
)
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

# Leer datos desde Gold
df_ml = spark.table("workspace.default.gold_telco")

# Seleccionar variables
df_ml = df_ml.select(
    "tenure",
    "MonthlyCharges",
    "Contract",
    "ChurnLabel"
).dropna()

print("Registros para Machine Learning:", df_ml.count())

display(df_ml.limit(5))


In [ ]:

# Dividir datos: 80% entrenamiento, 20% prueba

train, test = df_ml.randomSplit(
    [0.8, 0.2],
    seed=42
)

print("Entrenamiento:", train.count())
print("Prueba:", test.count())



In [ ]:

# Convertir contrato a valores numéricos
indexer = StringIndexer(
    inputCol="Contract",
    outputCol="ContractIndex",
    handleInvalid="keep"
)

# Codificar tipo de contrato
encoder = OneHotEncoder(
    inputCols=["ContractIndex"],
    outputCols=["ContractVector"]
)

# Unir variables predictoras
assembler = VectorAssembler(
    inputCols=[
        "tenure",
        "MonthlyCharges",
        "ContractVector"
    ],
    outputCol="features"
)

# Modelo de regresión logística
lr = LogisticRegression(
    featuresCol="features",
    labelCol="ChurnLabel",
    maxIter=100
)

# Crear pipeline
pipeline = Pipeline(
    stages=[indexer, encoder, assembler, lr]
)

# Entrenar solamente con train
modelo = pipeline.fit(train)

print("MODELO ENTRENADO CORRECTAMENTE")


In [ ]:

# Evaluar con datos que el modelo no vio

predicciones = modelo.transform(test)

display(
    predicciones.select(
        "tenure",
        "MonthlyCharges",
        "Contract",
        "ChurnLabel",
        "prediction",
        "probability"
    ).limit(10)
)


In [ ]:

# Evaluar AUC
evaluador_auc = BinaryClassificationEvaluator(
    labelCol="ChurnLabel",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

auc = evaluador_auc.evaluate(predicciones)

# Evaluar Accuracy
evaluador_accuracy = MulticlassClassificationEvaluator(
    labelCol="ChurnLabel",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluador_accuracy.evaluate(predicciones)

# Evaluar F1
evaluador_f1 = MulticlassClassificationEvaluator(
    labelCol="ChurnLabel",
    predictionCol="prediction",
    metricName="f1"
)

f1 = evaluador_f1.evaluate(predicciones)

print("RESULTADOS DEL MODELO")
print("--------------------------")
print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC ROC: {auc:.4f}")


In [ ]:

# Matriz de confusión

matriz = predicciones.groupBy(
    "ChurnLabel",
    "prediction"
).count().orderBy(
    "ChurnLabel",
    "prediction"
)

display(matriz)


## Análisis descriptivo

In [ ]:

from pyspark.sql import functions as F

# Leer datos de la capa Gold
df = spark.table("workspace.default.gold_telco")

# 1. Abandono por contrato
graf_contrato = (
    df.groupBy("Contract")
    .agg(
        F.count("*").alias("TotalClientes"),
        F.sum("ChurnLabel").alias("Abandonaron"),
        F.round(
            F.avg("ChurnLabel") * 100, 2
        ).alias("PorcentajeAbandono")
    )
    .orderBy(F.desc("PorcentajeAbandono"))
)

display(graf_contrato)


In [ ]:

graf_antiguedad = (
    df.withColumn(
        "GrupoAntiguedad",
        F.when(F.col("tenure") <= 12, "0-12 meses")
         .when(F.col("tenure") <= 24, "13-24 meses")
         .when(F.col("tenure") <= 48, "25-48 meses")
         .otherwise("49+ meses")
    )
    .groupBy("GrupoAntiguedad")
    .agg(
        F.count("*").alias("TotalClientes"),
        F.round(
            F.avg("ChurnLabel") * 100, 2
        ).alias("PorcentajeAbandono")
    )
)

display(graf_antiguedad)


In [ ]:

graf_internet = (
    df.groupBy("InternetService")
    .agg(
        F.count("*").alias("TotalClientes"),
        F.round(
            F.avg("ChurnLabel") * 100, 2
        ).alias("PorcentajeAbandono")
    )
    .orderBy(F.desc("PorcentajeAbandono"))
)

display(graf_internet)


In [ ]:

from pyspark.sql import functions as F

# Leer datos de la capa Gold
df = spark.table("workspace.default.gold_telco")

# Agrupar clientes por antigüedad
df_tiempo = df.withColumn(
    "Periodo",
    F.when(F.col("tenure") <= 12, "01. 0-12 meses")
     .when(F.col("tenure") <= 24, "02. 13-24 meses")
     .when(F.col("tenure") <= 36, "03. 25-36 meses")
     .when(F.col("tenure") <= 48, "04. 37-48 meses")
     .when(F.col("tenure") <= 60, "05. 49-60 meses")
     .otherwise("06. 61-72 meses")
)

# Contar clientes que permanecen y abandonan
grafica = df_tiempo.groupBy("Periodo").agg(
    F.sum(
        F.when(F.col("ChurnLabel") == 0, 1).otherwise(0)
    ).alias("Permanecen"),
    F.sum(
        F.when(F.col("ChurnLabel") == 1, 1).otherwise(0)
    ).alias("Abandonan")
).orderBy("Periodo")

display(grafica)


## Registro y recuperación con MLflow

In [ ]:

import mlflow
import mlflow.spark

print("Versión de MLflow:", mlflow.__version__)

mlflow.set_registry_uri("databricks-uc")

print("Registro de modelos:", mlflow.get_registry_uri())


In [ ]:

from pyspark.ml import PipelineModel

modelos = [
    nombre
    for nombre, objeto in list(globals().items())
    if isinstance(objeto, PipelineModel)
]

print("Modelos entrenados encontrados:", modelos)


In [ ]:

import os
import mlflow
import mlflow.spark
from mlflow.models import infer_signature

# Configurar Unity Catalog
mlflow.set_registry_uri("databricks-uc")

# Almacenamiento temporal para Spark ML
ruta_temporal = "/Volumes/workspace/default/telco_churn/mlflow_tmp"

os.makedirs(ruta_temporal, exist_ok=True)

os.environ["MLFLOW_DFS_TMP"] = ruta_temporal

# Datos de entrada
entrada = spark.table("workspace.default.gold_telco").select(
    "tenure",
    "MonthlyCharges",
    "Contract"
).limit(10)

# Firma del modelo
firma = infer_signature(
    entrada.toPandas(),
    modelo.transform(entrada).select("prediction").toPandas()
)

# Registrar modelo y métricas
with mlflow.start_run(run_name="Telco_Churn_Logistic_Regression"):

    mlflow.log_metric("accuracy", 0.7842)
    mlflow.log_metric("f1_score", 0.7741)
    mlflow.log_metric("auc_roc", 0.8217)

    modelo_registrado = mlflow.spark.log_model(
        spark_model=modelo,
        artifact_path="modelo_telco",
        registered_model_name="workspace.default.telco_churn_model",
        signature=firma,
        dfs_tmpdir=ruta_temporal
    )

print("Registro del modelo finalizado")


In [ ]:

import mlflow
import mlflow.spark

mlflow.set_registry_uri("databricks-uc")

# Recuperar la version 1 del modelo registrado
modelo_publicado = mlflow.spark.load_model(
    "models:/workspace.default.telco_churn_model/1",
    dfs_tmpdir="/Volumes/workspace/default/telco_churn/mlflow_tmp"
)

# Consultar los datos de la capa Gold
datos = spark.table("workspace.default.gold_telco")

# Generar predicciones
predicciones = modelo_publicado.transform(datos)

# Mostrar resultados
display(
    predicciones.select(
        "customerID",
        "tenure",
        "Contract",
        "MonthlyCharges",
        "ChurnLabel",
        "prediction",
        "probability"
    ).limit(20)
)


## Predicciones y niveles de riesgo

En este notebook: bajo < 40%, medio desde 40%, alto desde 70%.

In [ ]:

from pyspark.sql import functions as F
from pyspark.ml.functions import vector_to_array

# Extraer probabilidad de abandono
resultados = predicciones.withColumn(
    "ProbabilidadAbandono",
    F.round(vector_to_array("probability")[1] * 100, 2)
)

# Clasificar el nivel de riesgo
resultados = resultados.withColumn(
    "NivelRiesgo",
    F.when(F.col("ProbabilidadAbandono") >= 70, "Alto")
     .when(F.col("ProbabilidadAbandono") >= 40, "Medio")
     .otherwise("Bajo")
)

# Interpretar la prediccion
resultados = resultados.withColumn(
    "Prediccion",
    F.when(F.col("prediction") == 1, "Abandona")
     .otherwise("Permanece")
)

# Seleccionar columnas relevantes
resultados = resultados.select(
    "customerID",
    "tenure",
    "Contract",
    "MonthlyCharges",
    "ChurnLabel",
    "Prediccion",
    "ProbabilidadAbandono",
    "NivelRiesgo"
)

# Guardar tabla en Unity Catalog
resultados.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.gold_predicciones"
)

# Mostrar clientes con mayor riesgo
display(
    resultados.orderBy(
        F.desc("ProbabilidadAbandono")
    ).limit(20)
)

print("Total de registros:", resultados.count())
